# AHBA-GTEx End-to-End Write-up Workflow

This notebook is the canonical interactive entrypoint for the current repository. It starts from a raw `gxp_samples.csv` file, runs the shared preprocessing and the three manuscript comparators (naive fill, DLAM, PLAM), performs the all-gene evaluation workflow, and builds manuscript-facing tables and figures.


In [2]:

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

if "__file__" in globals():
    REPO_ROOT = Path(__file__).resolve().parents[1]
else:
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.workflows import (
    build_manuscript_assets,
    load_dataset_bundle,
    load_workflow_config,
    run_allgene_loro,
    run_dlam,
    run_naive_fill,
    run_plam,
    select_representative_subject,
    summarize_shared_harmonization,
    write_run_manifest,
)


def _parse_cli() -> argparse.Namespace:
    p = argparse.ArgumentParser(add_help=False)
    p.add_argument("--config", default=str(REPO_ROOT / "configs" / "notebook_full.yaml"))
    return p.parse_known_args()[0]


ARGS = _parse_cli() if "__file__" in globals() else argparse.Namespace(config=str(REPO_ROOT / "configs" / "notebook_full.yaml"))
CONFIG_PATH = Path(ARGS.config)
CFG = load_workflow_config(CONFIG_PATH)
if not Path(str(CFG.get("csv_path", ""))).is_absolute():
    CFG["csv_path"] = str((REPO_ROOT / str(CFG.get("csv_path"))).resolve())
if not Path(str(CFG.get("hvg_path", ""))).is_absolute():
    CFG["hvg_path"] = str((REPO_ROOT / str(CFG.get("hvg_path"))).resolve())
if not Path(str(CFG.get("out_root", ""))).is_absolute():
    CFG["out_root"] = str((REPO_ROOT / str(CFG.get("out_root"))).resolve())
if not Path(str(CFG.get("manuscript_root", ""))).is_absolute():
    CFG["manuscript_root"] = str((REPO_ROOT / str(CFG.get("manuscript_root"))).resolve())

print("Config:")
print(json.dumps({k: v for k, v in CFG.items() if k != "force_rerun_stages"}, indent=2))

Config:
{
  "csv_path": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/data/raw/gxp_samples.csv",
  "hvg_path": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/data/raw/ahba_100hvg.txt",
  "out_root": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup",
  "manuscript_root": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/docs/manuscript",
  "write_manuscript_assets": true,
  "export_full_harmonized_csv": false,
  "smoke_subjects": 0,
  "chunk_size": 500,
  "n_jobs": 1,
  "seed": 123,
  "combat_use_covariates": true,
  "min_observed_parcels": 5,
  "c_min": 8,
  "n_comp_target": 3,
  "ridge_alpha_bridge": 0.01,
  "rbf_smoothing": 0.1,
  "gp_length_scale": 25.0,
  "gp_noise": 0.001,
  "latent_dim": 3,
  "plam_max_iters": 5,
  "lambda_w": 1.0,
  "lambda_z": 1.0,
  "lambda_cal_a": 10.0,
  "lambda_cal_b": 10.0,
  "robust_loss": "student_t",
  "heteroscedastic": true,
  "calibration_mode": "hier_affine_map",
  "uncertainty_shrink": false,
  "dlam_basis_m

## 1. Load and validate the dataset bundle


In [9]:
BUNDLE = load_dataset_bundle(CFG["csv_path"], CFG["hvg_path"], CFG)
summary_df = pd.DataFrame([BUNDLE.counts])
summary_df


,n_genes_all,n_genes_hvg,n_ahba_rows,n_gtex_rows,n_subjects_total,n_subjects_eligible,n_parcels
0,13618,88,553,3030,385,313,150


## 2. Shared harmonization summary


In [10]:
HARMONIZATION = summarize_shared_harmonization(BUNDLE, CFG)
pd.DataFrame([HARMONIZATION["summary"]])


,n_subjects_eligible,n_genes_hvg,combat_use_covariates
0,313,88,True


## 3. Naive atlas-fill baseline


In [11]:
NAIVE_RESULT = run_naive_fill(BUNDLE, CFG)
pd.DataFrame([NAIVE_RESULT.summary])


,n_genes_hvg,n_parcels,n_subjects
0,88,150,313


## 4. DLAM


In [12]:
DLAM_RESULT = run_dlam(BUNDLE, CFG)
pd.DataFrame([DLAM_RESULT.summary])


,n_subjects,n_parcels,n_genes_hvg,compat_root
0,313,150,88,/scratch/asr655/neuroinformatics/Seq2GeneEx/gt...


In [13]:
DLAM_RESULT

ModelRunResult(model_name='dlam', stage_name='dlam', cache_dir='/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/dlam', manifest_path='/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/manifests/dlam.json', outputs={'atlas_mean_harmonized_csv': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/dlam/compat_root/tables/aggregate_allgenes_combat__affine_gl3__constrained_anchor__rbf_mean_harmonized.csv', 'subject_predictions_npz': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/dlam/subject_predictions_hvg.npz'}, summary={'n_subjects': 313, 'n_parcels': 150, 'n_genes_hvg': 88, 'compat_root': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/dlam/compat_root'})

## 5. PLAM


In [14]:
PLAM_RESULT = run_plam(BUNDLE, CFG)
pd.DataFrame([PLAM_RESULT.summary])


,n_subjects,n_parcels,n_genes_hvg,compat_root
0,313,150,88,/scratch/asr655/neuroinformatics/Seq2GeneEx/gt...


In [16]:
PLAM_RESULT

ModelRunResult(model_name='plam', stage_name='plam', cache_dir='/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/plam', manifest_path='/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/manifests/plam.json', outputs={'combined_harmonized_csv': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/plam/compat_export_root/gxp_completed_harmonized_combined.csv', 'subject_predictions_npz': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/plam/subject_predictions_hvg.npz', 'atlas_mean_harmonized_csv': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/plam/plam_atlas_mean_harmonized.csv'}, summary={'n_subjects': 313, 'n_parcels': 150, 'n_genes_hvg': 88, 'compat_root': '/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/notebook_writeup/caches/plam/compat_export_root'})

## 6. All-gene LORO evaluation


In [ ]:
EVAL_NAIVE = run_allgene_loro(BUNDLE, "naive", None, CFG)
EVAL_DLAM = run_allgene_loro(BUNDLE, "dlam", None, CFG)
EVAL_PLAM = run_allgene_loro(BUNDLE, "plam", None, CFG)
EVAL_RESULTS = {"naive": EVAL_NAIVE, "dlam": EVAL_DLAM, "plam": EVAL_PLAM}
summary_table = pd.read_csv(EVAL_PLAM.tables["summary_csv"])
summary_table


[allgene loro] 1/313 GTEX-1117F obs=8
[allgene loro] 2/313 GTEX-111FC obs=10
[allgene loro] 3/313 GTEX-117XS obs=11
[allgene loro] 4/313 GTEX-1192X obs=10
[allgene loro] 5/313 GTEX-11DXW obs=9
[allgene loro] 6/313 GTEX-11DXY obs=10
[allgene loro] 7/313 GTEX-11DYG obs=9
[allgene loro] 8/313 GTEX-11DZ1 obs=8
[allgene loro] 9/313 GTEX-11EI6 obs=8
[allgene loro] 10/313 GTEX-11EMC obs=8
[allgene loro] 11/313 GTEX-11GS4 obs=10
[allgene loro] 12/313 GTEX-11GSO obs=11
[allgene loro] 13/313 GTEX-11GSP obs=10
[allgene loro] 14/313 GTEX-11H98 obs=7
[allgene loro] 15/313 GTEX-11NUK obs=8
[allgene loro] 16/313 GTEX-11NV4 obs=10
[allgene loro] 17/313 GTEX-11O72 obs=8
[allgene loro] 18/313 GTEX-11OF3 obs=9
[allgene loro] 19/313 GTEX-11ONC obs=9
[allgene loro] 20/313 GTEX-11PRG obs=10
[allgene loro] 21/313 GTEX-11TTK obs=10
[allgene loro] 22/313 GTEX-11UD1 obs=8
[allgene loro] 23/313 GTEX-11WQC obs=8
[allgene loro] 24/313 GTEX-11ZTS obs=9
[allgene loro] 25/313 GTEX-11ZU8 obs=5
[allgene loro] 26/313 GT

## 7. Representative subject


In [ ]:
REPRESENTATIVE_SUBJECT = select_representative_subject(EVAL_RESULTS, CFG)
pd.DataFrame([{
    "subject": REPRESENTATIVE_SUBJECT.subject,
    "n_obs_parcels": REPRESENTATIVE_SUBJECT.n_obs_parcels,
    "selection_rule": REPRESENTATIVE_SUBJECT.selection_rule,
    **REPRESENTATIVE_SUBJECT.metadata,
}])


## 8. Manuscript assets


In [ ]:
ASSETS = build_manuscript_assets(BUNDLE, {"naive": NAIVE_RESULT, "dlam": DLAM_RESULT, "plam": PLAM_RESULT}, EVAL_RESULTS, CFG)
pd.DataFrame({"kind": ["figures", "tables"], "count": [len(ASSETS.figure_paths), len(ASSETS.table_paths)], "target": [ASSETS.manuscript_fig_dir, ASSETS.manuscript_table_dir]})


## 9. Final manifest


In [ ]:
RUN_MANIFEST = write_run_manifest(BUNDLE, {"naive": NAIVE_RESULT, "dlam": DLAM_RESULT, "plam": PLAM_RESULT}, EVAL_RESULTS, ASSETS, CFG)
pd.DataFrame([{
    "config_hash": RUN_MANIFEST["config_hash"],
    "timestamp": RUN_MANIFEST["timestamp"],
    "n_subjects_eligible": RUN_MANIFEST["n_subjects_eligible"],
    "n_genes_all": RUN_MANIFEST["n_genes_all"],
    "n_genes_hvg": RUN_MANIFEST["n_genes_hvg"],
    "representative_subject": REPRESENTATIVE_SUBJECT.subject,
}])


## 10. Key output locations


In [ ]:
print("Run manifest:", Path(BUNDLE.out_root) / "manifests" / "run_manifest.json")
print("Evaluation figures:", Path(EVAL_PLAM.asset_root) / "figures")
print("Evaluation tables:", Path(EVAL_PLAM.asset_root) / "tables")
print("Manuscript figure dir:", ASSETS.manuscript_fig_dir)
print("Manuscript table dir:", ASSETS.manuscript_table_dir)
